In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.bronze_superstore
USING DELTA
AS
SELECT 
    `Row ID` as row_id,
    `Order ID` as order_id,
    `Order Date` as order_date,
    `Ship Date` as ship_date,
    `Ship Mode` as ship_mode,
    `Customer ID` as customer_id,
    `Customer Name` as customer_name,
    Segment,
    Country,
    City,
    State,
    `Postal Code` as postal_code,
    Region,
    `Product ID` as product_id,
    Category,
    `Sub-Category` as sub_category,
    `Product Name` as product_name,
    Sales as vlr_sales,
    Quantity,
    Discount as vlr_discount,
    Profit as vlr_profit,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

-- Dimensão produto
CREATE OR REPLACE TABLE er_tech.superstore.bronze_produto
USING DELTA
AS
SELECT DISTINCT
    `Product ID`   as product_id,
    `Product Name` as product_name,
    `Category`     as category,
    `Sub-Category` as sub_category,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

-- -- Dimensão customer
CREATE OR REPLACE TABLE er_tech.superstore.bronze_customer
USING DELTA
AS
SELECT DISTINCT
    `Customer ID`   as customer_id,
    `Customer Name` as customer_name,
    `Segment`       as segment,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

-- -- Dimensão location
CREATE OR REPLACE TABLE er_tech.superstore.bronze_location
USING DELTA
AS
SELECT DISTINCT
    `Postal Code`  as postal_code,
    `Region`       as region_name,
    `City`         as city,
    `State`        as state,
    `Country`      as country,
    CURRENT_TIMESTAMP() AS load_date,
    'superstore_csv' AS source_system
FROM er_tech.superstore.tb_superstore
;

## Test Layer

In [0]:
%sql
-- Teste realizado para verificar se numero do produto pode ser utilizado como chave.
with cte_product_adjust as (
    select 
        right(product_id, 8) as product_id_number
    from 
        er_tech.superstore.bronze_produto
)

select
    product_id_number,
    count(product_id_number)
from cte_product_adjust 
group by 
    product_id_number
having count(product_id_number) > 1
;

select * from er_tech.superstore.bronze_produto where product_id like '%10000240%';

In [0]:
%sql
-- WITH CTE_LOCA AS (
--         SELECT DISTINCT
--         --ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
--         --regexp_replace(customer_id, '[^a-zA-Z0-9]', '') as customer_id,
--         postal_code,
--         region_name,
--         city,
--         state,
--         country,
--         CURRENT_TIMESTAMP() AS created_at,
--         --CURRENT_TIMESTAMP() AS updated_at
--     FROM 
--         er_tech.superstore.bronze_location
--     ;
-- )

---
select *
FROM 
    er_tech.superstore.gold_superstore
where customer_sk = 1
    and order_id = 'CA2017147039'
;

In [0]:
%sql
select * from er_tech.superstore.bronze_superstore

## Camada silver

In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.silver_produto
    USING DELTA
    AS
        select 
            ROW_NUMBER() OVER (ORDER BY product_id) AS product_sk,
            regexp_replace(product_id, '[^a-zA-Z0-9]', '') as product_id,
            product_name,
            category,
            sub_category,
            CURRENT_TIMESTAMP() AS created_at,
            CURRENT_TIMESTAMP() AS updated_at
        from 
            er_tech.superstore.bronze_produto
        ;

In [0]:
%sql
-- CREATE OR REPLACE TABLE er_tech.superstore.silver_customer
--     USING DELTA
--     AS
    SELECT
        ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
        regexp_replace(customer_id, '[^a-zA-Z0-9]', '') as customer_id,
        customer_name,
        segment,
        CURRENT_TIMESTAMP() AS created_at,
        CURRENT_TIMESTAMP() AS updated_at
    FROM 
        er_tech.superstore.bronze_customer
    ;

In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.silver_location
   USING DELTA
    AS
    SELECT DISTINCT
        ROW_NUMBER() OVER (ORDER BY postal_code, state, city) AS location_sk,
        postal_code,
        region_name,
        city,
        state,
        country,
        CURRENT_TIMESTAMP() AS created_at,
        CURRENT_TIMESTAMP() AS updated_at
    FROM 
        er_tech.superstore.bronze_location
    ;

In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.silver_superstore
   USING DELTA
    AS
    SELECT 
        row_id,
        regexp_replace(order_id, '[^a-zA-Z0-9]', '') as order_id,
        order_date,
        ship_date,
        ship_mode,
        regexp_replace(customer_id, '[^a-zA-Z0-9]', '') as customer_id,
        customer_name,
        Segment,
        Country,
        City,
        State,
        postal_code,
        Region,
        regexp_replace(product_id, '[^a-zA-Z0-9]', '') as product_id,
        Category,
        sub_category,
        product_name,
        vlr_sales,
        quantity,
        vlr_discount,
        vlr_profit,
        CURRENT_TIMESTAMP() AS created_at,
        CURRENT_TIMESTAMP() AS updated_at
    from er_tech.superstore.bronze_superstore;

## Camada Gold

In [0]:
%sql
CREATE OR REPLACE TABLE er_tech.superstore.gold_superstore
   USING DELTA
   as 
    SELECT DISTINCT
        c.customer_sk,
        p.product_sk,
        l.location_sk,
        s.ship_mode,
        s.order_date,
        s.ship_date,
        s.order_id,
        s.vlr_sales,
        s.Quantity,
        s.vlr_discount,
        s.vlr_profit
    FROM er_tech.superstore.silver_superstore s
    LEFT JOIN er_tech.superstore.silver_customer c
        ON s.customer_id = c.customer_id

    LEFT JOIN er_tech.superstore.silver_produto p
        ON s.product_id = p.product_id --and try_cast(s.product_name as string) and try_cast(p.product_name as string)

    LEFT JOIN er_tech.superstore.silver_location l
        ON s.Country = l.country
        AND s.State = l.state
        AND s.City = l.city
        AND S.postal_code = L.postal_code
        ;

In [0]:
%sql
select * from er_tech.superstore.silver_customer;

In [0]:
%sql
select 
    c.customer_name,
    round(sum(vlr_sales),2) as total_sales 
from 
    er_tech.superstore.gold_superstore d
    left join er_tech.superstore.silver_customer c on d.customer_sk = c.customer_sk 
group by 
    c.customer_name
order by 
    total_sales desc
limit 5
;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select 
    p.product_name,
    round(sum(vlr_sales),2) as total_sales 
from 
    er_tech.superstore.gold_superstore d
    left join er_tech.superstore.silver_produto p on d.customer_sk = p.product_sk
group by 
    p.product_name
order by 
    total_sales desc
limit 5
;

In [0]:
%sql
select 
    p.city,
    round(sum(vlr_sales),2) as total_sales 
from 
    er_tech.superstore.gold_superstore d
    left join er_tech.superstore.silver_location p on d.location_sk = p.location_sk
group by 
    p.city
having total_sales > 25000
order by total_sales desc
;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select * from er_tech.superstore.silver_location;